# Quick self-play evaluation

Compare selected checkpoints against all-greedy play on 200 shared states after six throws. The model controls only turn 7, turn 8, or turn 9 in separate runs; greedy controls the other turns. Each reported advantage is paired by state. This configuration completed in 27 seconds in the current environment.

In [ ]:
import sys
from pathlib import Path
from time import perf_counter

repo_root = Path.cwd().resolve()
while not (repo_root / 'data_generation.py').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Could not find repository root')
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import self_play_training as self_play

In [ ]:
seed = 2026
num_states = 200  # 100 paired states per model-turn group
grid_size = (10, 10, 4)
max_stones = 10

weights_root = repo_root / 'weights' / 'self_play'
checkpoint_paths = {
    'large_m30': weights_root / 'round9_20k' / 'checkpoints' / 'm_30.npz',
    'small_m10': weights_root / 'round9_2k_v2' / 'checkpoints' / 'm_10.npz',
}

In [ ]:
started_at = perf_counter()
results = self_play.evaluate_checkpoints_against_greedy(
    checkpoint_paths=checkpoint_paths,
    num_states=num_states,
    seed=seed,
    grid_size=grid_size,
    max_stones=max_stones,
)
elapsed_seconds = perf_counter() - started_at
print(f'evaluation elapsed: {elapsed_seconds:.1f} seconds')

In [ ]:
for row in results:
    print(
        f"{row['run']:<10} {row['group']:<13} "
        f"advantage={row['average_advantage']:+.3f} "
        f"95% CI=[{row['advantage_ci95_low']:+.3f}, "
        f"{row['advantage_ci95_high']:+.3f}] "
        f"win/tie/loss={row['win_rate']:.0%}/"
        f"{row['tie_rate']:.0%}/{row['loss_rate']:.0%}"
    )